# FARSITE Single-Step Demo

Fully self-contained single FARSITE simulation step.
No outputs from other notebooks are required.

**Steps:**
1. Define fire parameters
2. Fetch perimeters from WIFIRE
3. Download LANDFIRE + generate `landscape.lcp` (skipped if already exists)
4. Fetch weather from WIFIRE
5. Run FARSITE (t=0 → t=1)
6. Compare prediction vs observed

**Prerequisites:** `TestFARSITE`, `lcpmake`, and `install_packages.sh` must be executable:
```bash
chmod +x TestFARSITE lcpmake install_packages.sh
```

In [ ]:
!./install_packages.sh

**Important:** After running `install_packages.sh`, switch to the IPython kernel named `Python (gdal_env)`.

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import zipfile
import io
import subprocess
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.validation import make_valid
from osgeo import gdal, osr

gdal.UseExceptions()
osr.UseExceptions()

# Ensure working directory is the notebook directory
os.chdir(Path(os.path.abspath('')))

from farsite import forward_pass_farsite, DEFAULT_DIST_RES, DEFAULT_PERIM_RES, DEFAULT_TEMPERATURE, DEFAULT_HUMIDITY      

print('Imports OK')
print(f'Working directory: {Path.cwd()}')

## 2. Define Fire Parameters

In [ ]:
FIRE_NAME          = 'Border 2 Synthetic'
GEOSERVER_LAYER    = 'WIFIRE:synthetic_fire_perimeters'
IGNITION_LAT       =  32.61    # WGS84 — used for LANDFIRE download area
IGNITION_LON       = -116.88
RADIUS_KM          = 10        # LANDFIRE download radius (km)
LANDFIRE_EMAIL     = 'h7ahmed@ucsd.edu'   # ← change this
LCP_PATH           = Path('landscape.lcp')
DIST_RES           = DEFAULT_DIST_RES   # meters
PERIM_RES          = DEFAULT_PERIM_RES  # meters

print(f'Fire:    {FIRE_NAME}')
print(f'LCP:     {LCP_PATH}')
print(f'Dist/Perim res: {DIST_RES}/{PERIM_RES} m')

## 3. Fetch Fire Perimeters

In [ ]:
def fetch_fire_perimeters(fire_name, geoserver_layer, verbose=True):
    """Fetch fire perimeters ingested into WIFIRE Firemap WFS from FIRIS, sorted oldest to newest."""
    params = {
        'SERVICE':      'WFS',
        'VERSION':      '2.0.0',
        'REQUEST':      'GetFeature',
        'TYPENAMES':    geoserver_layer,
        'CQL_FILTER':   f"fire_name = '{fire_name}'",
        'OUTPUTFORMAT': 'application/json',
        'SRSNAME':      'EPSG:4326',
    }
    if verbose:
        print(f"Fetching perimeters for '{fire_name}'...")
    resp = requests.get('https://firemap.sdsc.edu/geoserver/wfs', params=params, timeout=30)
    resp.raise_for_status()
    features = resp.json().get('features', [])
    if not features:
        raise ValueError(f"No perimeters found for '{fire_name}'")

    gdf = gpd.GeoDataFrame.from_features(features, crs='EPSG:4326')
    gdf['datetime'] = pd.to_datetime(gdf['perimeter_timestamp'].str.rstrip('Z'))

    # MultiPolygon -> largest polygon
    def largest_poly(geom):
        if isinstance(geom, MultiPolygon):
            return max(geom.geoms, key=lambda g: g.area)
        return geom
    gdf['geometry'] = gdf['geometry'].apply(largest_poly)
    gdf = gdf.sort_values('datetime').reset_index(drop=True)
    gdf = gdf.to_crs('EPSG:5070')

    if verbose:
        print(f'  Retrieved {len(gdf)} perimeters:')
        for i, row in gdf.iterrows():
            print(f'  [{i}] {row["datetime"]}  —  {row.geometry.area / 1e6:.2f} km²')
    return gdf

In [ ]:
perimeters_gdf = fetch_fire_perimeters(FIRE_NAME, GEOSERVER_LAYER)

if len(perimeters_gdf) < 2:
    raise ValueError('Need at least 2 perimeters')

# t=0: used as FARSITE ignition input
# t=1: used as ground truth for comparison
t0_row = perimeters_gdf.iloc[0]
t1_row = perimeters_gdf.iloc[1]

initial_poly  = t0_row.geometry
observed_poly = t1_row.geometry
T0            = pd.Timestamp(t0_row['datetime'])
T1            = pd.Timestamp(t1_row['datetime'])
DT            = T1 - T0

# Weather query location = centroid of first perimeter in WGS84
centroid_wgs84 = gpd.GeoSeries([initial_poly], crs='EPSG:5070').to_crs('EPSG:4326').centroid.iloc[0]
WX_LAT = centroid_wgs84.y
WX_LON = centroid_wgs84.x

print(f'\nIgnition (t=0): {T0}  —  {initial_poly.area / 1e6:.3f} km²')
print(f'Observed (t=1): {T1}  —  {observed_poly.area / 1e6:.3f} km²')
print(f'Interval:       {DT}')
print(f'Weather loc:    ({WX_LAT:.4f}, {WX_LON:.4f})')

## 4. Download LANDFIRE + Generate Landscape File
Skipped automatically if `landscape.lcp` already exists.

In [ ]:
def download_landfire_data(poly, output_dir, email, verbose=True):
    """
    Download LANDFIRE raster data for the bounding box of a polygon.
    poly should be in EPSG:4326.
    Returns dict of paths to per-layer ASCII rasters.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    # Buffer slightly so the domain is larger than the ignition point
    buffered = poly.buffer(0.5, cap_style='flat', join_style='bevel')
    minx, miny, maxx, maxy = buffered.bounds

    if verbose:
        print(f'Submitting LANDFIRE request...')
        print(f'  Bounding box: [{minx:.4f}, {miny:.4f}, {maxx:.4f}, {maxy:.4f}]')

    params = {
        'Email':               email,
        'Layer_List':          '250CBD;250CBH;250CC;250CH;250FBFM40;ASP2020;ELEV2020;SLPP2020',
        'Area_of_Interest':    f'{minx} {miny} {maxx} {maxy}',
        'Output_Projection':   '5070',
        'Resample_Resolution': '90',
        'Priority_Code':       'K3LS9F',
    }
    resp = requests.get('https://lfps.usgs.gov/api/job/submit', params=params, timeout=30)
    resp.raise_for_status()
    job_id = resp.json()['jobId']
    if verbose:
        print(f'  Job ID: {job_id}')

    t_start = time.time()
    while True:
        s = requests.get(f'https://lfps.usgs.gov/api/job/status?JobId={job_id}', timeout=30).json()
        status = s.get('status', '').lower()
        if verbose:
            print(f'  [{int(time.time()-t_start)}s] {status}')
        if status == 'succeeded':
            download_url = s['outputFile']
            break
        elif status in ('failed', 'canceled'):
            raise RuntimeError(f'LANDFIRE job {status}')
        time.sleep(10)

    zip_resp = requests.get(download_url, stream=True, timeout=60)
    zip_resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(zip_resp.content)) as zf:
        zf.extractall(output_dir)

    multi_tif = next(output_dir.glob('*.tif'))
    layer_names = ['250CBD','250CBH','250CC','250CH','250FBFM40','ASP2020','ELEV2020','SLPP2020']
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for band_idx, name in enumerate(layer_names, start=1):
            gdal.Translate(str(output_dir / f'{name}.asc'), str(multi_tif),
                           format='AAIGrid', bandList=[band_idx])
            if verbose:
                print(f'  ✓ {name}.asc')

    return {
        'elevation':      output_dir / 'ELEV2020.asc',
        'slope':          output_dir / 'SLPP2020.asc',
        'aspect':         output_dir / 'ASP2020.asc',
        'fuel':           output_dir / '250FBFM40.asc',
        'canopy_cover':   output_dir / '250CC.asc',
        'canopy_height':  output_dir / '250CH.asc',
        'canopy_base':    output_dir / '250CBH.asc',
        'canopy_density': output_dir / '250CBD.asc',
    }


def generate_lcp_from_rasters(output_path, rasters, latitude, verbose=True):
    """Run lcpmake to build a FARSITE landscape (.lcp) file from ASCII rasters."""
    lcpmake = Path(os.path.abspath('lcpmake'))
    if not lcpmake.exists():
        raise FileNotFoundError(f'lcpmake not found at {lcpmake}')

    output_path = Path(output_path)
    cmd = [
        str(lcpmake),
        '-latitude',  str(latitude),
        '-landscape', str(output_path.with_suffix('')),
        '-elevation', str(rasters['elevation']),
        '-slope',     str(rasters['slope']),
        '-aspect',    str(rasters['aspect']),
        '-fuel',      str(rasters['fuel']),
        '-cover',     str(rasters['canopy_cover']),
        '-height',    str(rasters['canopy_height']),
        '-base',      str(rasters['canopy_base']),
        '-density',   str(rasters['canopy_density']),
        '-fb40',
    ]
    if verbose:
        print('Running lcpmake...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'lcpmake failed:\n{result.stderr}')
    lcp = output_path.with_suffix('.lcp')
    if not lcp.exists():
        raise RuntimeError('lcpmake ran but .lcp was not created')
    if verbose:
        print(f'  ✓ {lcp}  ({lcp.stat().st_size / 1024:.1f} KB)')
    return lcp

In [ ]:
if LCP_PATH.exists():
    print(f'✓ Using existing landscape file: {LCP_PATH}  ({LCP_PATH.stat().st_size / 1024:.1f} KB)')
else:
    print('Landscape file not found — downloading LANDFIRE data...')

    # Build a bounding polygon in WGS84 around the ignition point
    ignition_pt  = Point(IGNITION_LON, IGNITION_LAT)
    pt_gdf       = gpd.GeoSeries([ignition_pt], crs='EPSG:4326')
    ignition_poly_wgs84 = pt_gdf.to_crs(pt_gdf.estimate_utm_crs()) \
                                 .buffer(RADIUS_KM * 1000) \
                                 .to_crs('EPSG:4326').iloc[0]

    rasters = download_landfire_data(
        poly=ignition_poly_wgs84,
        output_dir=Path('data') / 'landfire',
        email=LANDFIRE_EMAIL,
    )
    LCP_PATH = generate_lcp_from_rasters(
        output_path=LCP_PATH,
        rasters=rasters,
        latitude=IGNITION_LAT,
    )

print(f'LCP ready: {LCP_PATH}')

## 5. Fetch Weather

**Note:** This example only retrieves ``wind_speed`` and ``wind_direction``, but additional weather observables that can be retrieved and used as FARSITE inputs include ``temperature`` and ``humidity``.

In [ ]:
def fetch_weather(lat, lon, start_dt, end_dt, verbose=True):
    """
    Fetch mean wind speed and direction from WIFIRE Firemap pylaski API.
    Returns dict with 'windspeed' and 'winddirection'.
    """
    start_str = pd.Timestamp(start_dt).strftime('%Y-%m-%dT%H:%M:%S')
    end_str   = pd.Timestamp(end_dt).strftime('%Y-%m-%dT%H:%M:%S')

    timestamp = int(time.time() * 1000)
    params = {
        'selection':  'closestTo',
        'lat':        str(lat),
        'lon':        str(lon),
        'observable': ['wind_speed', 'wind_direction'],
        'from':       start_str,
        'to':         end_str,
        'callback':   'wxData',
        '_':          str(timestamp),
    }
    if verbose:
        print(f'  Querying weather: {start_str} to {end_str}')
    try:
        resp = requests.get('https://firemap.sdsc.edu/pylaski/stations/data',
                            params=params, timeout=15)
        resp.raise_for_status()
        text = resp.text.strip()
        if text.startswith('wxData(') and text.endswith(')'):
            text = text[len('wxData('):-1]
        data    = json.loads(text)
        station = data['features'][0]['properties']
        ws_list = station.get('wind_speed', [])
        wd_list = station.get('wind_direction', [])
        ws = float(np.mean(ws_list)) if ws_list else 5.0
        wd = float(np.mean(wd_list)) if wd_list else 270.0
        if verbose:
            print(f'  Wind: {ws:.1f} mph @ {wd:.0f}°')
        return {'windspeed': ws, 'winddirection': wd}
    except Exception as e:
        if verbose:
            print(f'  ⚠ Weather fetch failed ({e}), using defaults: 5 mph @ 270°')
        return {'windspeed': 5.0, 'winddirection': 270.0}

In [ ]:
weather = fetch_weather(WX_LAT, WX_LON, T0, T1)
print(f'Wind speed:     {weather["windspeed"]:.1f} mph')
print(f'Wind direction: {weather["winddirection"]:.0f}°')

## 6. Run FARSITE

In [ ]:
farsite_params = {
    'windspeed':     int(weather['windspeed']),
    'winddirection': int(weather['winddirection']),
    'temperature' : int(DEFAULT_TEMPERATURE),
    'humidity' : int(DEFAULT_HUMIDITY),
    'dt':            DT.to_pytimedelta(),
}

print('Running FARSITE...')
print(f'  Period:  {T0}  →  {T1}  ({DT})')
print(f'  Wind:    {farsite_params["windspeed"]} mph @ {farsite_params["winddirection"]}°')
print(f'  Temperature:    {farsite_params["temperature"]}')
print(f'  Humidity:    {farsite_params["humidity"]}')
print(f'  Input:   {initial_poly.area / 1e6:.3f} km²')

predicted_poly = forward_pass_farsite(
    poly=initial_poly,
    params=farsite_params,
    start_time=T0.strftime('%Y-%m-%d %H:%M:%S'),
    lcppath=str(LCP_PATH),
    dist_res=DIST_RES,
    perim_res=PERIM_RES,
)

if predicted_poly is not None:
    print(f'\n✓ FARSITE succeeded')
    print(f'  Predicted area: {predicted_poly.area / 1e6:.3f} km²')
    print(f'  Observed area:  {observed_poly.area / 1e6:.3f} km²')
else:
    print('\n✗ FARSITE failed — check tmp/logs/')

## 7. Compare Prediction vs Observed

In [ ]:
if predicted_poly is None:
    print('No output to visualize.')
else:
    obs  = make_valid(observed_poly)
    pred = make_valid(predicted_poly)
    try:
        intersection = obs.intersection(pred).area
        union        = obs.union(pred).area
    except Exception:
        obs  = obs.buffer(0)
        pred = pred.buffer(0)
        intersection = obs.intersection(pred).area
        union        = obs.union(pred).area

    iou          = intersection / union if union > 0 else 0.0
    area_err_pct = abs(pred.area - obs.area) / obs.area * 100

    print(f'Observed area:  {obs.area / 1e6:.3f} km²')
    print(f'Predicted area: {pred.area / 1e6:.3f} km²')
    print(f'Area error:     {area_err_pct:.1f}%')
    print(f'IoU:            {iou:.3f}')

    try:
        import contextily as ctx
        use_basemap = True
    except ImportError:
        use_basemap = False

    fig, ax = plt.subplots(figsize=(10, 10))

    gpd.GeoSeries([pred], crs='EPSG:5070').to_crs('EPSG:3857').boundary.plot(
        ax=ax, color='#e74c3c', linewidth=2.5)
    gpd.GeoSeries([obs], crs='EPSG:5070').to_crs('EPSG:3857').boundary.plot(
        ax=ax, color='#3498db', linewidth=2.5)

    if use_basemap:
        try:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, crs='EPSG:3857')
        except Exception as e:
            print(f'Basemap skipped: {e}')

    ax.legend(handles=[
        Patch(facecolor='none', edgecolor='#e74c3c',
              label=f'FARSITE prediction  {pred.area/1e6:.3f} km²'),
        Patch(facecolor='none', edgecolor='#3498db',
              label=f'Observed            {obs.area/1e6:.3f} km²'),
    ], loc='upper left', fontsize=10)

    ax.set_title(
        f'{FIRE_NAME}\n'
        f'{T0.strftime("%Y-%m-%d %H:%M")} → {T1.strftime("%H:%M")}  |  '
        f'Wind: {farsite_params["windspeed"]} mph @ {farsite_params["winddirection"]}°  |  '
        f'IoU: {iou:.3f}  |  Area error: {area_err_pct:.1f}%',
        fontsize=11
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig('farsite_demo_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved → farsite_demo_result.png')